# Agent pipeline walkthrough

Exercises every piece of the agent pipeline end to end:

1. `risk_node` — cancellation-risk ranking, reachable via the router
2. `validate_node` — deterministic grounding check before any answer ships
3. `judge_node` — sampled LLM-as-judge groundedness check
4. `log_node` — structured logging of every request to Postgres (`agent_logs`)
5. `review_queue` + `agent/review.py` — human review loop, promotable into the eval golden set
6. `agent/eval.py` — offline regression harness
7. LangSmith tracing (opt-in via env vars, zero code changes)

**Before running**: select the project's `.venv` as the kernel (Python 3.12, the one created by `uv sync`), and make sure `docker compose up -d --wait` is running so Postgres is reachable. `.env` (repo root) is loaded automatically the first time anything imports the `agent` package (see `agent/__init__.py`), so no manual env setup is needed here.

In [1]:
import sys
from pathlib import Path

# repo root on sys.path (so `import agent`, `import ml` work regardless of where
# jupyter was launched from) — .env loading itself is automatic (python-dotenv,
# see agent/__init__.py), triggered by the `from agent import ...` in the next cell.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

In [2]:
import os

# force a high judge sample rate for this notebook run, so the judge_node
# demos below actually fire instead of being skipped by chance (default is
# 0.3 in production — see agent/graph.py JUDGE_SAMPLE_RATE)
os.environ["JUDGE_SAMPLE_RATE"] = "1.0"

import pandas as pd

from agent import db, llm, review
from agent.graph import AGENT, State, log_node, validate_node

print("llm.PROVIDER resolved to:", llm.PROVIDER)
print("LANGCHAIN_TRACING_V2:", os.environ.get("LANGCHAIN_TRACING_V2"))
print("LANGCHAIN_PROJECT:", os.environ.get("LANGCHAIN_PROJECT"))
assert llm.PROVIDER == "groq", (
    "expected a real provider (groq) so judge_node/tracing sections below have "
    "something to exercise — check .env (repo root) has GROQ_API_KEY/LLM_PROVIDER set"
)

llm.PROVIDER resolved to: groq
LANGCHAIN_TRACING_V2: true
LANGCHAIN_PROJECT: property-insights-assistant


## 0. Graph shape

`START -> router -> {sql_node | rag_node | risk_node} -> validate_node -> judge_node -> log_node -> END`

In [3]:
try:
    print(AGENT.get_graph().draw_mermaid())
except Exception as e:
    print(f"(mermaid render unavailable: {e})")
    for node in AGENT.get_graph().nodes:
        print("-", node)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	sql_node(sql_node)
	rag_node(rag_node)
	risk_node(risk_node)
	validate_node(validate_node)
	judge_node(judge_node)
	log_node(log_node)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	judge_node --> log_node;
	rag_node --> validate_node;
	risk_node --> validate_node;
	router -. &nbsp;rag&nbsp; .-> rag_node;
	router -. &nbsp;risk&nbsp; .-> risk_node;
	router -. &nbsp;sql&nbsp; .-> sql_node;
	sql_node --> validate_node;
	validate_node --> judge_node;
	log_node --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 1. `risk_node` is reachable

Previously: the router's `classify_intent` already returned `"risk"` for risk-shaped questions, but `route()` only allowed `sql`/`rag` through, and `risk_node` was never `add_node`'d into the graph — so risk questions silently fell back to `sql_node`, which doesn't know what to do with them.

Run a genuine risk question and confirm the full pipeline (DB → ML model → retrieval-for-notes) actually executes.

In [4]:
out = AGENT.invoke({"question": "which Lisbon listings are most at risk of cancellation?"})

assert out["intent"] == "risk", f"expected intent=risk, got {out['intent']!r}"
assert out["answer"].startswith("[RISK]"), "risk_node should tag its answer"
assert out["risk_result"], "risk_node should populate risk_result with scored listings"

print("intent:", out["intent"])
print("validation_issues:", out["validation_issues"])
print()
print(out["answer"])

intent: risk
validation_issues: []

[RISK] Highest-risk listings:
L0029: risk=64% — Long-term repeat guests; very stable booking history.
L0011: risk=59% — Building lift has been unreliable; two recent guests cancelled citing 4th-floor access.
L0079: risk=51% — Well-reviewed; guests routinely rebook the following year.
L0001: risk=51% — History of winter no-shows; three guests cancelled last-minute in Jan-Feb.
L0012: risk=48% — Long-term repeat guests; very stable booking history.


## 2. `validate_node` — grounding check before an answer ships

Deterministic, no extra LLM call: pulls listing IDs (`L\d{4}`) out of the final answer and checks they're a subset of what was actually retrieved (`rag_context`) or scored (`risk_result`). Two cases below: a clean answer (should pass silently), and a hand-crafted state with a fabricated citation (should be caught).

In [5]:
# 2a. clean case — real rag_node output, should have zero validation_issues
clean = AGENT.invoke({"question": "why might some Lisbon listings be problematic?"})
print("intent:", clean["intent"])
print("validation_issues:", clean["validation_issues"])
print("answer:", clean["answer"])
assert clean["validation_issues"] == [], "expected a real, grounded answer to pass validation"

intent: rag
validation_issues: []
answer: Some Lisbon listings might be problematic due to issues such as: 
1. History of winter no-shows (L0001) 
2. Unreliable building lift, leading to guest cancellations (L0011)


In [6]:
# 2b. adversarial case — hand-craft a state where the "answer" cites a listing
# (L9999) that was never in the retrieved context. validate_node should flag it.
fabricated: State = {
    "question": "why might some Lisbon listings be problematic?",
    "intent": "rag",
    "rag_context": "L0001 (Lisbon): history of winter no-shows.",
    "answer": "L0001 has issues, and L9999 is also notoriously unreliable.",
}
checked = validate_node(fabricated)

print("validation_issues:", checked["validation_issues"])
print("answer:", checked["answer"])
assert checked["validation_issues"], "expected the fabricated L9999 citation to be caught"
assert "[UNVERIFIED]" in checked["answer"], "flagged answers should be annotated before shipping"

validation_issues: ["cites listing(s) not in retrieved context: ['L9999']"]
answer: L0001 has issues, and L9999 is also notoriously unreliable.

[UNVERIFIED] cites listing(s) not in retrieved context: ['L9999']


## 3. `judge_node` — sampled LLM-as-judge

Catches a subtler failure mode than ID-hallucination: an answer that cites a *real*, retrieved listing but overstates or misreads what its context actually says. Only runs for `rag`/`risk` intents, only with a real provider (skipped entirely under `LLM_PROVIDER=offline` — nothing to judge with), and normally only samples a fraction of traffic (`JUDGE_SAMPLE_RATE`, forced to `1.0` above so this cell reliably exercises it).

In [7]:
judged = AGENT.invoke({"question": "what makes this listing special?"})

print("intent:", judged["intent"])
print("judge_result:", judged["judge_result"])
assert judged["judge_result"] is not None, (
    "expected judge_node to fire (real provider + rag intent + sample_rate=1.0)"
)
assert "grounded" in judged["judge_result"]

intent: rag
judge_result: {'grounded': True, 'reason': 'The answer states a lack of information.'}


In [10]:
# under LLM_PROVIDER=offline (what tests/CI use), judge_node should be a no-op
prev_provider = llm.PROVIDER
llm.PROVIDER = "offline"
try:
    offline_out = AGENT.invoke({"question": "what makes this listing special?"})
    print("judge_result under offline provider:", offline_out["judge_result"])
    assert offline_out["judge_result"] is None
finally:
    llm.PROVIDER = prev_provider  # restore for later cells

judge_result under offline provider: None


## 4. `log_node` — structured logging to Postgres

Every graph run (regardless of entry point — CLI, eval, this notebook) writes a row to `agent_logs` with latency, provider, and validation/judge outcomes. Query it back and look at the rows the cells above just produced.

In [12]:
rows = db.query("""
    SELECT id, intent, provider, latency_ms, judge_grounded, needs_review, created_at
    FROM agent_logs ORDER BY id DESC LIMIT 10
""")
logs_df = pd.DataFrame(rows, columns=[
    "id", "intent", "provider", "latency_ms", "judge_grounded", "needs_review", "created_at",
])
logs_df

,id,intent,provider,latency_ms,judge_grounded,needs_review,created_at
0,158,rag,offline,2.314091,None,False,2026-07-20 21:42:25.655188+00:00
1,157,rag,offline,2.875090,None,False,2026-07-20 21:39:29.163130+00:00
2,156,rag,groq,878.226995,True,False,2026-07-20 21:38:33.780013+00:00
3,155,rag,groq,1003.621101,True,False,2026-07-20 21:37:26.549990+00:00
4,154,risk,groq,908.159018,False,True,2026-07-20 21:37:06.377870+00:00
5,153,risk,offline,133.895874,None,False,2026-07-20 21:05:45.001286+00:00
6,152,rag,offline,1.153708,None,False,2026-07-20 21:05:44.864582+00:00
7,151,sql,offline,5.769014,None,False,2026-07-20 21:05:44.860721+00:00
8,150,risk,offline,537.663937,None,False,2026-07-20 21:05:44.848831+00:00
9,149,risk,offline,140.518188,None,False,2026-07-20 21:05:44.306327+00:00


## 5. `review_queue` + `agent/review.py`

Anything `validate_node` or `judge_node` flags lands in `review_queue`. `agent/review.py` exposes `list` / `promote` / `resolve`. Section 2b already showed `validate_node` catching a fabricated citation — reuse that same fabricated state, but this time push it through `log_node` too (not just `validate_node`), so it actually reaches `review_queue`, the way a real flagged production request would.

(Note: pushing a *rigged context* through the real `AGENT.invoke(...)` pipeline — e.g. monkeypatching `retriever.retrieve` to return a context that doesn't mention any problem — was tried first, but the real model correctly declined to fabricate a citation from it and `validation_issues` came back empty. That's actually a good sign for `validate_node`'s design: it isn't needed to catch a well-behaved model. To reliably demonstrate the review-queue path regardless of model behavior, we construct the flagged state directly instead.)

All rows/cases created by this notebook are tagged `NOTEBOOK_TEST:` and cleaned up in the last cell — safe to re-run.

In [19]:
# the fabricated state from 2b, tagged NOTEBOOK_TEST: so cleanup can find it,
# pushed through validate_node -> log_node (the real chain that reaches
# review_queue) instead of validate_node alone.
flagged: State = {
    "question": "NOTEBOOK_TEST: why might some Lisbon listings be problematic?",
    "intent": "rag",
    "rag_context": "L0001 (Lisbon): history of winter no-shows.",
    "answer": "L0001 has issues, and L9999 is also notoriously unreliable.",
}
flagged = validate_node(flagged)
flagged = log_node(flagged)

print("validation_issues:", flagged["validation_issues"])
print("answer:", flagged["answer"])

validation_issues: ["cites listing(s) not in retrieved context: ['L9999']"]
answer: L0001 has issues, and L9999 is also notoriously unreliable.

[UNVERIFIED] cites listing(s) not in retrieved context: ['L9999']


In [20]:
pending = db.query("""
    SELECT id, question, reason, status FROM review_queue
    WHERE question LIKE 'NOTEBOOK_TEST:%' ORDER BY id DESC
""")
review_df = pd.DataFrame(pending, columns=["id", "question", "reason", "status"])
review_df

,id,question,reason,status
0,27,NOTEBOOK_TEST: why might some Lisbon listings ...,cites listing(s) not in retrieved context: ['L...,pending


In [21]:
# promote it into the eval golden set, same as a human reviewer would after
# confirming this is a real failure worth guarding against permanently
if len(review_df):
    review_id = int(review_df.iloc[0]["id"])
    review.promote(review_id, intent="rag", contains=None)

    import json
    with open(review.CASES_PATH) as f:
        cases = json.load(f)
    print("last case in eval_cases.json:", cases[-1])

Marked review 27 resolved.
Promoted to /Users/ali/Git_Projects/ai-engineer-interview-main/agent/eval_cases.json: {'question': 'NOTEBOOK_TEST: why might some Lisbon listings be problematic?', 'expect_intent': 'rag'}
last case in eval_cases.json: {'question': 'NOTEBOOK_TEST: why might some Lisbon listings be problematic?', 'expect_intent': 'rag'}


## 6. `agent/eval.py` — offline regression harness

Runs the golden set (now including the promoted case above) through the graph and checks router accuracy + grounding + expected markers. Exits non-zero on failure — CI-gateable as-is.

In [22]:
import importlib

from agent import eval as agent_eval

importlib.reload(agent_eval)  # pick up the case we just promoted into the JSON file

all_passed = agent_eval.run()

result intent     ms  question
----------------------------------------------------------------------
PASS   sql       231  what is the average price in Lisbon?
PASS   sql       164  how many confirmed bookings do we have?
PASS   sql       166  what is the cancellation rate in Porto?
PASS   rag       892  why might some Lisbon listings be problematic?
PASS   rag       679  what makes this listing special?
PASS   risk      745  which Lisbon listings are most at risk of cancellation?
PASS   risk     1265  which listings are most likely to cancel?
PASS   rag      1082  NOTEBOOK_TEST: why might some Lisbon listings be problematic?
----------------------------------------------------------------------
ALL PASSED


## 7. LangSmith tracing

Native to LangGraph via env vars — no code changes. `langsmith` is already installed as a transitive dependency of `langgraph` (verified: `ls .venv/.../site-packages | grep langsmith`). If `LANGCHAIN_TRACING_V2=true` and a valid `LANGCHAIN_API_KEY` were loaded in the setup cell, every `AGENT.invoke(...)` call above should already have produced a trace.

This notebook can't check the dashboard for you (no browser access from here) — open **smith.langchain.com** → project `property-insights-assistant` and confirm runs are showing up with per-node timing.

In [ ]:
tracing_on = (
    os.environ.get("LANGCHAIN_TRACING_V2") == "true"
    and bool(os.environ.get("LANGCHAIN_API_KEY"))
)
print("Tracing should be active:", tracing_on)
print("Project:", os.environ.get("LANGCHAIN_PROJECT"))
print("Check: https://smith.langchain.com")

## Cleanup

Remove everything this notebook wrote (tagged `NOTEBOOK_TEST:`) from Postgres and roll back the promoted eval case, so re-running this notebook — or running `agent.eval` / `agent.review` for real afterwards — starts from a clean slate.

In [18]:
import json

db.query("DELETE FROM review_queue WHERE question LIKE 'NOTEBOOK_TEST:%'")
db.query("DELETE FROM agent_logs WHERE question LIKE 'NOTEBOOK_TEST:%'")

with open(review.CASES_PATH) as f:
    cases = json.load(f)
cases = [c for c in cases if not c["question"].startswith("NOTEBOOK_TEST:")]
with open(review.CASES_PATH, "w") as f:
    json.dump(cases, f, indent=2)
    f.write("\n")

print("Cleaned up. Remaining eval cases:", len(cases))

Cleaned up. Remaining eval cases: 7
